In [1]:
from pathlib import Path
import pandas as pd
import json
import time
import sys

In [2]:
PROJECT_ROOT = Path("/home/harielpadillasanchez/Documentos/TT/TT2")
DATA_DIR = PROJECT_ROOT / "data"
SPLIT_DIR = DATA_DIR / "splits"
SFT_DIR = DATA_DIR / "sft_ready"
SFT_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = "feina_clean_30"

TRAIN_PATH = SPLIT_DIR / f"{PREFIX}_train.csv"
VAL_PATH   = SPLIT_DIR / f"{PREFIX}_val.csv"
TEST_PATH  = SPLIT_DIR / f"{PREFIX}_test.csv"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


In [3]:
df_train = pd.read_csv(TRAIN_PATH)
df_val = pd.read_csv(VAL_PATH)
df_test = pd.read_csv(TEST_PATH)

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)

Train: (1110, 15)
Val: (238, 15)
Test: (238, 15)


In [4]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = " ".join(text.split())
    return text.strip()

for df_ in [df_train, df_val, df_test]:
    df_["source_text"] = df_["source_text"].apply(clean_text)
    df_["reference_text"] = df_["reference_text"].apply(clean_text)

In [5]:
def drop_invalid_rows(df_):
    before = len(df_)
    df_ = df_.copy()

    df_ = df_[
        (df_["source_text"].str.len() > 0) &
        (df_["reference_text"].str.len() > 0)
    ].copy()

    after = len(df_)
    print(f"Filas antes: {before} | después: {after} | eliminadas: {before - after}")
    return df_

df_train = drop_invalid_rows(df_train)
df_val = drop_invalid_rows(df_val)
df_test = drop_invalid_rows(df_test)

Filas antes: 1110 | después: 1110 | eliminadas: 0
Filas antes: 238 | después: 238 | eliminadas: 0
Filas antes: 238 | después: 238 | eliminadas: 0


In [6]:
from configs.prompts import ZERO_SHOT_TEMPLATE

def build_sft_prompt(source_text: str, rules_block: str = "") -> str:
    return ZERO_SHOT_TEMPLATE.format(
        rules_block=rules_block,
        source=source_text,
    )

In [7]:
def build_sft_dataframe(df_, rules_block: str = ""):
    df_sft = df_.copy()

    df_sft["instruction"] = df_sft["source_text"].apply(
        lambda x: build_sft_prompt(x, rules_block=rules_block)
    )
    df_sft["output"] = df_sft["reference_text"]

    df_sft["text"] = df_sft.apply(
        lambda row: row["instruction"] + row["output"],
        axis=1
    )

    return df_sft

train_sft = build_sft_dataframe(df_train, rules_block="")
val_sft = build_sft_dataframe(df_val, rules_block="")
test_sft = build_sft_dataframe(df_test, rules_block="")

In [8]:
display(
    train_sft[["row_id", "instruction", "output", "text"]].head(2)
)

,row_id,instruction,output,text
0,8,Reescribe en español el siguiente texto con le...,Se integró un equipo profesional interdiscipli...,Reescribe en español el siguiente texto con le...
1,12,Reescribe en español el siguiente texto con le...,"Se puede afirmar que el ser humano, la familia...",Reescribe en español el siguiente texto con le...


In [9]:
for split_name, df_ in [("train", train_sft), ("val", val_sft), ("test", test_sft)]:
    df_["instruction_chars"] = df_["instruction"].str.len()
    df_["output_chars"] = df_["output"].str.len()
    df_["text_chars"] = df_["text"].str.len()

    print(f"\n--- {split_name.upper()} ---")
    print(df_[["instruction_chars", "output_chars", "text_chars"]].describe())


--- TRAIN ---
       instruction_chars  output_chars   text_chars
count        1110.000000   1110.000000  1110.000000
mean          382.856757    137.778378   520.635135
std            97.885960     80.516970   171.680489
min           225.000000     10.000000   238.000000
25%           314.000000     81.000000   397.000000
50%           359.000000    120.000000   483.000000
75%           425.000000    175.000000   603.500000
max           825.000000    497.000000  1256.000000

--- VAL ---
       instruction_chars  output_chars   text_chars
count         238.000000    238.000000   238.000000
mean          378.147059    136.399160   514.546218
std            93.122347     77.981797   166.448089
min           225.000000     14.000000   239.000000
25%           319.250000     87.250000   414.500000
50%           359.000000    117.500000   475.500000
75%           413.750000    165.500000   585.000000
max           937.000000    587.000000  1482.000000

--- TEST ---
       instruction_cha

In [10]:
keep_cols = [
    "row_id",
    "instruction",
    "output",
    "text",
]

train_sft_final = train_sft[keep_cols].copy()
val_sft_final = val_sft[keep_cols].copy()
test_sft_final = test_sft[keep_cols].copy()

print(train_sft_final.shape, val_sft_final.shape, test_sft_final.shape)

(1110, 4) (238, 4) (238, 4)


In [11]:
TRAIN_SFT_CSV = SFT_DIR / "feina_30_train_sft.csv"
VAL_SFT_CSV   = SFT_DIR / "feina_30_val_sft.csv"
TEST_SFT_CSV  = SFT_DIR / "feina_30_test_sft.csv"

train_sft_final.to_csv(TRAIN_SFT_CSV, index=False, encoding="utf-8-sig")
val_sft_final.to_csv(VAL_SFT_CSV, index=False, encoding="utf-8-sig")
test_sft_final.to_csv(TEST_SFT_CSV, index=False, encoding="utf-8-sig")

print(TRAIN_SFT_CSV)
print(VAL_SFT_CSV)
print(TEST_SFT_CSV)

/home/harielpadillasanchez/Documentos/TT/TT2/data/sft_ready/feina_30_train_sft.csv
/home/harielpadillasanchez/Documentos/TT/TT2/data/sft_ready/feina_30_val_sft.csv
/home/harielpadillasanchez/Documentos/TT/TT2/data/sft_ready/feina_30_test_sft.csv


In [12]:
TRAIN_SFT_JSONL = SFT_DIR / "feina_30_train_sft.jsonl"
VAL_SFT_JSONL   = SFT_DIR / "feina_30_val_sft.jsonl"
TEST_SFT_JSONL  = SFT_DIR / "feina_30_test_sft.jsonl"

def save_jsonl(df_, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df_.iterrows():
            record = {
                "row_id": int(row["row_id"]),
                "instruction": row["instruction"],
                "output": row["output"],
                "text": row["text"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

save_jsonl(train_sft_final, TRAIN_SFT_JSONL)
save_jsonl(val_sft_final, VAL_SFT_JSONL)
save_jsonl(test_sft_final, TEST_SFT_JSONL)

print(TRAIN_SFT_JSONL)
print(VAL_SFT_JSONL)
print(TEST_SFT_JSONL)

/home/harielpadillasanchez/Documentos/TT/TT2/data/sft_ready/feina_30_train_sft.jsonl
/home/harielpadillasanchez/Documentos/TT/TT2/data/sft_ready/feina_30_val_sft.jsonl
/home/harielpadillasanchez/Documentos/TT/TT2/data/sft_ready/feina_30_test_sft.jsonl


Para el ajuste fino supervisado se reutilizó la estructura base de instrucción empleada en la fase zero-shot, con el fin de mantener consistencia entre las condiciones experimentales. De esta forma, la comparación entre prompting y LoRA se centra en el efecto del ajuste fino del modelo, y no en cambios adicionales en la formulación de la tarea.

proximos experimentos:
- LoRA base con ZERO_SHOT_TEMPLATE
- LoRA guiado con ZERO_SHOT_TEMPLATE + rules_block
